# Module 3B — Stock Screener

Once the optimizer decides **how much** to put in equity, this module decides **which stocks**.

**Screening factors:**
- **Beta** — volatility relative to Nifty 50 (low = stable, high = aggressive)
- **PE ratio** — valuation sanity check
- **Debt/Equity** — financial health
- **6-month momentum** — avoid falling stocks

Universe: ~80 curated Indian stocks across large, mid, and small cap tiers.

In [ ]:
!pip install -q numpy pandas yfinance plotly

In [ ]:
import sys, json
sys.path.insert(0, '../src')

from stock_screener import fetch_stock_universe, screen_stocks, build_equity_basket

In [ ]:
# Load risk profile from Module 1 (or set manually)
try:
    with open('../data/risk_profile.json') as f:
        profile = json.load(f)
    persona = profile['persona']
    print(f'Loaded profile: {persona}')
except FileNotFoundError:
    persona = 'Balanced'  # fallback
    print(f'No saved profile found, using: {persona}')

## Fetch and screen stocks
*(Takes 2–3 minutes — pulling live data for the full universe)*

In [ ]:
universe = fetch_stock_universe(persona)
screened = screen_stocks(persona, universe)
print(f'\nTop stocks for {persona} investor:')
print(screened[['name','tier','beta','pe','de','momentum_6m','score']].to_string())

## Build equity basket
Assuming optimizer put 45% of portfolio into equity (change this once Module 3 runs):

In [ ]:
# Replace 45.0 with actual equity % from optimizer output
equity_allocation_pct = 45.0

basket = build_equity_basket(screened, equity_allocation_pct)
print(f'\nEquity basket ({equity_allocation_pct}% of total portfolio):')
print(basket[['name','tier','equity_weight_pct','portfolio_weight_pct']].to_string())

## Visualise the basket

In [ ]:
import plotly.graph_objects as go

tier_colors = {'large': '#1565C0', 'mid': '#2E7D32', 'small': '#E65100'}
colors = [tier_colors.get(t, '#888') for t in basket['tier']]

fig = go.Figure(go.Bar(
    x=basket['name'],
    y=basket['portfolio_weight_pct'],
    marker_color=colors,
    text=[f"{v:.1f}%" for v in basket['portfolio_weight_pct']],
    textposition='outside',
))
fig.update_layout(
    title=f'Equity Basket — {persona} Investor (total equity = {equity_allocation_pct}%)',
    xaxis_tickangle=-35,
    yaxis_title='% of Total Portfolio',
    template='plotly_white',
    height=500,
)
fig.show()

# Save basket
basket.to_csv('../data/equity_basket.csv', index=False)
print('Basket saved to data/equity_basket.csv')

## Screening logic — why these filters?

| Filter | Rationale |
|--------|-----------|
| **Beta** | Measures systematic risk. Conservative investors can't afford high market sensitivity — a beta of 0.7 means the stock moves 70% as much as the market. |
| **PE ratio** | Valuation check. Extremely high PE (>50) means you're paying a large premium for future growth — riskier if growth disappoints. |
| **Debt/Equity** | Highly leveraged companies amplify losses in downturns. Especially important for conservative profiles. |
| **6m momentum** | Behavioral finance finding: recent losers tend to keep losing short-term. We avoid stocks down >30% to sidestep value traps. |

**Limitation:** This is a factor-based screen, not a DCF valuation. It ranks stocks by risk-adjusted quality signals, not intrinsic value. A proper deep-dive on any individual name is always recommended before investing.